# 03 · Chat Models and Messages

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** Ollama. OpenAI / Anthropic cells
> run only if you've put their keys in `01_LANGGRAPH/.env`, and are skipped otherwise.

Nodes that think need a model. LangGraph itself doesn't talk to models: it runs
whatever your nodes do. In practice, nodes use LangChain's **chat model
interface**, one API that looks the same for Ollama, OpenAI, Anthropic and
dozens of other providers. This notebook teaches that interface, the
**message** types that flow through it, and how messages live inside graph state.

**You will learn**
- The chat-model interface: `invoke`, `stream`, `batch`, `with_structured_output`
- The four message types, and what's inside an `AIMessage`
- How to switch provider with one line, and what `common.get_model()` does
- **Structured output**: getting validated Pydantic objects instead of prose
- `MessagesState` and the `add_messages` reducer: append, edit, delete messages
- Keeping conversations inside the context window, and counting tokens

**Prerequisites:** notebooks 01–02.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

import os
from typing import Annotated, TypedDict, Literal
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

## 1 · Why a common interface matters

Every provider's raw API is different: different request shapes, different ways
to express tools, different streaming formats, different token-count fields. If
your nodes called those APIs directly, switching from a local model during
development to a cloud model in production would mean rewriting every node.

LangChain's chat models wrap each provider behind one interface:

```
                 ┌──────────────── BaseChatModel ────────────────┐
 your node ────► │ invoke · stream · batch · bind_tools ·        │
                 │ with_structured_output                        │
                 └──────┬───────────────┬───────────────┬────────┘
                        ▼               ▼               ▼
                   ChatOllama       ChatOpenAI     ChatAnthropic   ...
                  (local, free)      (cloud)         (cloud)
```

A node written against this interface runs unchanged on any of them.

## 2 · Connecting to a model

The direct way is to import the provider's class:

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model=os.getenv('OLLAMA_MODEL', 'qwen3-coder:30b'), temperature=0)
reply = model.invoke('What is the capital of the Netherlands? One word.')
print(type(reply).__name__, '→', reply.content)

The provider-neutral way is `init_chat_model("provider:model")`, which picks the
right class for you. This is how you make the provider a **configuration
value** rather than an import:

In [ ]:
from langchain.chat_models import init_chat_model

same_model = init_chat_model('ollama:qwen3-coder:30b', temperature=0)
print(type(same_model).__name__, '→', same_model.invoke('Say OK.').content)

This section's `common.py` wraps exactly that into `get_model()`. It reads
`MODEL_PROVIDER` and `<PROVIDER>_MODEL` from `01_LANGGRAPH/.env`, defaults to
Ollama, and raises a clear error if a cloud key is missing. Here's the whole thing:

In [ ]:
import inspect, common
print(inspect.getsource(common.get_model))

Now the same question to every provider you have configured. The node code
(`ask`) is identical for all of them:

In [ ]:
from common import get_model, has_key

def ask(m):
    return m.invoke('In exactly five words, what is LangGraph?').content

for provider in ('ollama', 'openai', 'anthropic'):
    if provider != 'ollama' and not has_key(provider):
        print(f'{provider:<10} (skipped: no {provider.upper()}_API_KEY in .env)')
        continue
    print(f'{provider:<10} {ask(get_model(provider))}')

**Common parameters** (names are normalised where possible):

| Parameter | Meaning |
|---|---|
| `temperature` | randomness. Use `0` for routing, extraction and tests; 0.7+ for creative text |
| `max_tokens` (`num_predict` on Ollama) | cap on the reply length |
| `timeout`, `max_retries` | network behaviour (cloud providers) |

## 3 · Messages: what goes in and what comes out

Chat models take a **list of messages** and return one message. There are four
types you'll use constantly:

| Type | Role | Created by | Key fields |
|---|---|---|---|
| `SystemMessage` | instructions / persona | you | `content` |
| `HumanMessage` | the user's turn | you | `content` |
| `AIMessage` | the model's turn | the model | `content`, `tool_calls`, `usage_metadata` |
| `ToolMessage` | a tool's result, sent back to the model | you (or `ToolNode`) | `content`, `tool_call_id` |

You can write them as classes or as `(role, text)` tuples. Both mean the same:

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

conversation = [
    SystemMessage('You are a terse assistant. Answer in one word.'),
    HumanMessage('Capital of France?'),
    AIMessage('Paris'),                      # a previous model turn (history)
    HumanMessage('And of Italy?'),
]
reply = model.invoke(conversation)
same = model.invoke([('system', 'You are a terse assistant. Answer in one word.'),
                     ('user', 'Capital of France?'), ('assistant', 'Paris'),
                     ('user', 'And of Italy?')])
print(reply.content, '|', same.content)

**The model has no memory.** It saw "And of Italy?" in context only because we
sent the whole conversation again. Every call is stateless; *you* (or the graph
state) carry the history. That's why message lists live in state.

Let's look inside an `AIMessage`, because nodes and routers read these fields:

In [ ]:
print('content        :', reply.content)
print('tool_calls     :', reply.tool_calls, ' ← filled when the model wants a tool (notebook 04)')
print('usage_metadata :', reply.usage_metadata)
print('id             :', reply.id)
print('response_meta  :', {k: reply.response_metadata[k] for k in ('model', 'done_reason')})

## 4 · Streaming and batching

- `stream()` yields **chunks** as the model generates them. Use it for user-facing
  UIs, where the first words appear immediately.
- `batch()` sends several independent prompts **concurrently**, which is faster
  than a loop of `invoke` calls.

In [ ]:
for chunk in model.stream('Count from 1 to 10, separated by spaces.'):
    print(chunk.content, end='|', flush=True)       # '|' shows the chunk boundaries
print()

import time
t0 = time.perf_counter()
answers = model.batch(['One word: a colour.', 'One word: an animal.', 'One word: a fruit.'])
print([a.content for a in answers], f'{time.perf_counter() - t0:.1f}s')

Inside a graph you rarely call `stream()` yourself. LangGraph can stream the
tokens of *any* model call in *any* node from the outside (notebook 09).

## 5 · Structured output: objects instead of prose

A router or an extractor can't act on "I think this is probably billing…". It
needs data. `with_structured_output(Schema)` returns a new runnable that makes
the model fill a Pydantic schema, and gives you a **validated object**:

In [ ]:
class Ticket(BaseModel):
    category: Literal['billing', 'technical', 'other'] = Field(description='the department')
    urgency: int = Field(ge=1, le=5, description=(
        '1 = question, no impact; 3 = something is broken; '
        '5 = customer is losing money now or faces a deadline within 48 hours'))
    summary: str = Field(description='one short sentence')


classifier = model.with_structured_output(Ticket)
t = classifier.invoke('I was charged three times this month and my rent is due tomorrow!!')
print(type(t).__name__, t)

You get a real `Ticket` object with a valid category, an int between 1 and 5,
and a summary, so code can branch on `t.category` safely. But look at the
**urgency**. For a customer charged three times with rent due tomorrow, the
description clearly says 5. Why didn't the model follow it?

**What each method sends to the model.** `with_structured_output` can work in
different ways, chosen with `method=`:

| `method=` | How the schema is used | Does the model *read* your descriptions? |
|---|---|---|
| `'json_schema'` (Ollama default) | constrains **decoding**: the model can only emit tokens that fit the schema | **not necessarily**. On Ollama the schema shapes the output but isn't shown to the model as text |
| `'function_calling'` | sent as a **tool definition**, which models are trained to read | yes |
| instructions in the prompt | you write the rubric in the system/user message | yes |

Let's test all three on the same ticket:

In [ ]:
TICKET = 'I was charged three times this month and my rent is due tomorrow!!'
RUBRIC = ('Classify support tickets. Urgency: 1 = question, no impact; 3 = something is broken; '
          '5 = customer is losing money now or faces a deadline within 48 hours.')

variants = {
    "json_schema (default)":    lambda: model.with_structured_output(Ticket).invoke(TICKET),
    "function_calling":         lambda: model.with_structured_output(Ticket, method='function_calling').invoke(TICKET),
    "json_schema + rubric":     lambda: model.with_structured_output(Ticket).invoke([('system', RUBRIC), ('user', TICKET)]),
}
for name, run in variants.items():
    t = run()
    print(f'{name:<24} urgency={t.urgency}  category={t.category}')

The **same schema** gives different answers depending on whether the model
actually saw the scale. The lessons:

- **Format ≠ meaning.** Constrained decoding guarantees the *shape* (valid JSON,
  allowed values, right types). It says nothing about whether the values are
  *right*.
- **Put the rubric where the model reads it.** Either in the prompt (works with
  every method and provider) or via `method='function_calling'`. Cloud
  providers generally pass the schema to the model with its descriptions, but a
  rubric in the prompt is the portable choice.
- **Test your classifiers on known cases** like this one. A classifier that
  always says "1" still produces perfectly valid objects.

Even the right method can fail validation sometimes (e.g. a value outside
`ge=1, le=5`). In production use `include_raw=True`: instead of raising, you get
the raw message, the parsed object (or `None`) and the error, so a node can
retry or fall back:

In [ ]:
classifier = model.with_structured_output(Ticket, method='function_calling')   # used below
safe_classifier = model.with_structured_output(Ticket, method='function_calling', include_raw=True)
out = safe_classifier.invoke('The app is a bit slow sometimes.')
print('keys          :', list(out))
print('parsed        :', out['parsed'])
print('parsing_error :', out['parsing_error'])

Design tips for schemas:
- **`Literal[...]` for anything you'll branch on**, so the model can't invent a category.
- **Descriptions are prompts only if the model sees them.** Use `function_calling` or repeat the rubric in the prompt.
- **Keep schemas small and flat.** Deeply nested schemas fail more often on small local models.

## 6 · Messages inside graph state

Conversational graphs keep the message list in state. Plain `operator.add` would
work for appending, but chat needs more: *editing* a message (e.g. a human
corrects the model's draft) and *deleting* old messages. LangGraph's
**`add_messages`** reducer does all three, matching messages by `id`:

In [ ]:
from langgraph.graph.message import add_messages
from langchain_core.messages import RemoveMessage

history = [HumanMessage('hi', id='m1'), AIMessage('hello!', id='m2')]

print('append :', [m.content for m in add_messages(history, [HumanMessage('how are you?')])])
print('edit   :', [m.content for m in add_messages(history, [AIMessage('Hello there!', id='m2')])])
print('delete :', [m.content for m in add_messages(history, [RemoveMessage(id='m1')])])

- A message with a **new id** (or none; one is generated) is **appended**.
- A message with an **existing id** **replaces** that message.
- `RemoveMessage(id=...)` **deletes** it.

`MessagesState` is a ready-made state with exactly one key,
`messages: Annotated[list, add_messages]`. Extend it when you need more keys.
Here's a minimal chatbot graph: one node that calls the model on the
conversation so far.

In [ ]:
from langgraph.graph import MessagesState


class ChatState(MessagesState):          # messages: Annotated[list, add_messages]  (inherited)
    turns: int


def chatbot(state: ChatState) -> dict:
    reply = model.invoke([SystemMessage('You are a friendly, concise tutor.')] + state['messages'])
    return {'messages': [reply], 'turns': state.get('turns', 0) + 1}   # add_messages APPENDS


b = StateGraph(ChatState)
b.add_node('chatbot', chatbot)
b.add_edge(START, 'chatbot')
b.add_edge('chatbot', END)
chat = b.compile()

state = chat.invoke({'messages': [('user', 'In one sentence: what is a reducer in LangGraph?')]})
for m in state['messages']:
    print(f'{m.type:>6}: {m.content}')

To continue this conversation, you'd pass the previous `messages` plus the new
user turn. Notebook 08 shows the proper way: a **checkpointer** keeps each
conversation's state by `thread_id`, so you only send the new message.

## 7 · Staying inside the context window, and counting tokens

Conversations grow without limit, but context windows don't, and every token is
re-sent (and, in the cloud, re-billed) on every call. `trim_messages` keeps the
most recent messages that fit a budget:

In [ ]:
from langchain_core.messages import trim_messages

long_history = [SystemMessage('You are helpful.')]
for i in range(1, 21):
    long_history += [HumanMessage(f'question {i} ' + 'blah ' * 20), AIMessage(f'answer {i}')]

trimmed = trim_messages(long_history, max_tokens=200, token_counter='approximate',
                        strategy='last',          # keep the most recent
                        include_system=True,      # never drop the system prompt
                        start_on='human')         # don't start on an orphaned AI reply
print(f'{len(long_history)} messages → {len(trimmed)} kept:', [m.content[:12] for m in trimmed])

Trimming in a node (before calling the model) keeps the **state** complete and
the **prompt** small. For long-lived agents, summarising old turns into one
message is the more thorough alternative.

**Counting tokens.** Every `AIMessage` carries `usage_metadata`. To total a whole
graph run, wrap it in `get_usage_metadata_callback()`, which aggregates per model:

In [ ]:
from langchain_core.callbacks import get_usage_metadata_callback

with get_usage_metadata_callback() as usage:
    chat.invoke({'messages': [('user', 'Name three uses of checkpoints, briefly.')]})
    classifier.invoke('My invoice total is wrong.')
print(usage.usage_metadata)

Multiply by your provider's price per token for a cost estimate (Ollama: $0).
Notebook 12 turns this into a per-node cost meter.

## 8 · Trade-offs: local vs cloud models

| | Ollama (local) | Cloud (OpenAI, Anthropic, …) |
|---|---|---|
| Cost | free after hardware | per token |
| Privacy | data never leaves the machine | sent to a provider |
| Quality on hard reasoning | good for mid-size models, weaker on the hardest tasks | strongest models available |
| Tool calling / structured output | depends on the model; test yours | mature, reliable |
| Speed | depends on your GPU | usually fast, but network latency |

A common setup: **develop on Ollama** (free, fast iteration), **run evaluations
on both**, and pick per node. A cheap model for routing and extraction, a strong
one for the hard reasoning. `get_model(provider)` makes that a per-node choice.

## 9 · Common mistakes

- **Parsing free text** with regexes to make decisions. Use structured output.
- **Assuming the model remembers.** It doesn't; send the history, or let a checkpointer carry it.
- **Returning the whole message list from a node** with `add_messages`. That's
  fine because ids dedupe, but return only the *new* messages; it's clearer and cheaper.
- **Building a new model object inside every node call.** Create it once, outside the node.
- **Hard-coding the provider in nodes.** Pass a model in, or use one config value.
- **Unbounded history.** Trim or summarise before the context window, or the bill, overflows.

## 10 · Check yourself

<details><summary><b>1.</b> Your node works on Ollama. What has to change to run it on Anthropic?</summary>

Only which model object it uses (e.g. `get_model('anthropic')` or `init_chat_model('anthropic:...')`). The node calls the common interface (`invoke`, `with_structured_output`, …), which every provider implements.
</details>

<details><summary><b>2.</b> Why does a follow-up question like "And of Italy?" work only if you send earlier messages?</summary>

Chat models are stateless: each call sees only the messages sent in that call. Conversation memory is the caller's job, which in LangGraph means the message list in state, persisted by a checkpointer.
</details>

<details><summary><b>3.</b> You need a router to choose between three branches. What should the model return, and how?</summary>

A structured object with a `Literal` field listing exactly the three branch names, via `with_structured_output`. That guarantees a value the router can return directly, with no text parsing. The criteria for choosing belong in the prompt (or a method where the model reads the schema), because constrained decoding alone guarantees the shape, not the judgement.
</details>

<details><summary><b>4.</b> With <code>add_messages</code>, how do you (a) append, (b) replace, (c) delete a message?</summary>

(a) Return a message with a new id (or no id). (b) Return a message with the same `id` as the existing one. (c) Return `RemoveMessage(id=...)`.
</details>

<details><summary><b>5.</b> When would you use <code>include_raw=True</code>?</summary>

When a parsing or validation failure must not crash the node. You get `raw`, `parsed` and `parsing_error`, so you can retry, repair, or fall back instead of raising.
</details>

<details><summary><b>6.</b> Why trim messages in the node rather than deleting them from state?</summary>

Trimming only what's *sent* keeps the full history in state (for audit, replay, or summarising later) while keeping each prompt within the context window and cost budget.
</details>

## Takeaway

Nodes talk to models through one interface (`invoke`, `stream`, `batch`,
`with_structured_output`), so the provider is just configuration. Models are
stateless: the conversation lives in **graph state** as messages, managed by the
`add_messages` reducer. Make decisions with **structured output**, keep prompts
inside the context window with trimming, and track tokens per run.

Next → `04_tools_and_react_agent.ipynb`: letting the model call tools, and
building the agent loop as a graph.